# Pipeline de Teste de Integração Fim-a-Fim - BusFlow

Este notebook executa o teste do pipeline de dados BusFlow, realizando:
1. Leitura dos dados tratados na S3 (via S3 Gateway Endpoint).
2. Gravação de log de execução no RDS PostgreSQL (VPC Privada).
3. Publicação de notificação no SNS (via VPC Endpoint) para notificar via e-mail.

In [ ]:
# Instalação do driver PostgreSQL
%pip install psycopg2-binary

In [ ]:
import boto3
import pandas as pd
import psycopg2
import getpass
from datetime import datetime

In [ ]:
# Configurações AWS Dinâmicas
region = 'us-east-1'
bucket_name = 'trusted-busflow-2026-2'
db_identifier = 'db-busflow'

print("Buscando informações da infraestrutura AWS...")

# 1. Buscar Host do RDS
rds_client = boto3.client('rds', region_name=region)
db_instances = rds_client.describe_db_instances(DBInstanceIdentifier=db_identifier)
db_host = db_instances['DBInstances'][0]['Endpoint']['Address']
db_port = db_instances['DBInstances'][0]['Endpoint']['Port']
print(f"RDS Host encontrado: {db_host}:{db_port}")

# 2. Buscar Tópico SNS
sns_client = boto3.client('sns', region_name=region)
topics = sns_client.list_topics()
topic_arn = None
for topic in topics['Topics']:
    if 'topico-processamento-csv' in topic['TopicArn'] or 'sns-topic-busflow' in topic['TopicArn']:
        topic_arn = topic['TopicArn']
        break

if topic_arn:
    print(f"Tópico SNS encontrado: {topic_arn}")
else:
    raise ValueError("Tópico SNS 'topico-processamento-csv' não encontrado!")

In [ ]:
# 3. Buscar o arquivo mais recente no Bucket Trusted
s3_client = boto3.client('s3')
print(f"Listando arquivos no bucket: {bucket_name}")
response = s3_client.list_objects_v2(Bucket=bucket_name)
contents = response.get('Contents', [])
csv_files = [c for c in contents if c['Key'].endswith('.csv')]

if not csv_files:
    raise ValueError(f"Nenhum arquivo CSV encontrado no bucket {bucket_name}!")

# Ordenar por data de modificação
csv_files.sort(key=lambda x: x['LastModified'], reverse=True)
latest_file_key = csv_files[0]['Key']
print(f"Arquivo mais recente encontrado: {latest_file_key} (Modificado em: {csv_files[0]['LastModified']})")

# Ler dados usando pandas
df = pd.read_csv(f"s3://{bucket_name}/{latest_file_key}")
print(f"Dados carregados com sucesso! Linhas: {len(df)}, Colunas: {list(df.columns)}")

In [ ]:
# 4. Solicitar a senha do banco e conectar
db_password = getpass.getpass("Digite a senha master do banco (rds_master_password): ")

conn = psycopg2.connect(
    host=db_host,
    port=db_port,
    database="busflowdb",
    user="postgres",
    password=db_password
)
print("Conectado ao PostgreSQL com sucesso!")

In [ ]:
# 5. Criar tabela de logs e registrar execução
cursor = conn.cursor()
cursor.execute("""
    CREATE TABLE IF NOT EXISTS pipeline_logs (
        id SERIAL PRIMARY KEY,
        timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        rows_processed INT,
        file_key VARCHAR(255),
        status VARCHAR(50)
    )
""")
conn.commit()

cursor.execute(
    "INSERT INTO pipeline_logs (rows_processed, file_key, status) VALUES (%s, %s, %s)",
    (len(df), latest_file_key, "SUCCESS")
)
conn.commit()
print("Log registrado na tabela 'pipeline_logs' com sucesso!")

# Consultar última inserção para validar
cursor.execute("SELECT * FROM pipeline_logs ORDER BY id DESC LIMIT 1")
row = cursor.fetchone()
print(f"Registro gravado: {row}")

cursor.close()
conn.close()

In [ ]:
# 6. Publicar mensagem de sucesso no SNS para envio de e-mail
message = f"""Pipeline de dados BusFlow concluído com sucesso!

Detalhes da execução:
- Arquivo processado: s3://{bucket_name}/{latest_file_key}
- Linhas processadas: {len(df)}
- Banco de dados: RDS PostgreSQL ({db_host})
- Tabela de log: pipeline_logs
- Horário de conclusão: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

response = sns_client.publish(
    TopicArn=topic_arn,
    Message=message,
    Subject="[BusFlow] Pipeline Executado com Sucesso!"
)

print(f"Notificação enviada com sucesso! MessageId: {response['MessageId']}")